In [1]:
# ============================================================
# 07_rf_global_all_infections_all_countries.ipynb
#
# Global pooled Random Forest:
#   - all infections
#   - all valid countries
#   - direct multioutput h=1..4
#
# Methodology:
#   - Train: 2014-2022
#   - Test: rolling-origin 2023
#   - One single global model trained once
#   - Inputs:
#       recent_0..recent_3
#       seasonal_52..seasonal_55
#       sin/cos week
#       disease dummy
#       country dummy
#   - Output:
#       y_1, y_2, y_3, y_4
#   - Evaluation:
#       explicit, auditable, same target calendar logic as baselines/RF/VARIMA
# ============================================================

import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.ensemble import RandomForestRegressor
from sklearn.multioutput import MultiOutputRegressor
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

from src.config import get_project_paths
from src.io_data import read_ari_ili
from src.gaps import gap_summary
from src.impute import impute_series_weekly
from src.calendars import extract_raw_weekly_series
from src.metrics import mae, rmse, mase_scale_seasonal, mase_from_mae

paths = get_project_paths(PROJECT_ROOT)
paths.results.mkdir(parents=True, exist_ok=True)


# ============================================================
# 1. Configuration
# ============================================================

MIN_OBS = 300
MAX_GAP = 8
M_SEASON = 52
HORIZONS = (1, 2, 3, 4)
MAX_H = max(HORIZONS)

MODEL_NAME = "rf_global_all_infections_all_countries"

RF_PARAMS = dict(
    n_estimators=400,
    max_depth=12,
    min_samples_leaf=3,
    random_state=42,
    n_jobs=-1
)

RECENT_LAGS = [0, 1, 2, 3]          # y_t, y_{t-1}, y_{t-2}, y_{t-3}
SEASONAL_LAGS = [52, 53, 54, 55]   # y_{t-52}, ..., y_{t-55}


# ============================================================
# 2. Load data and select valid countries
# ============================================================

ari_df, ili_df = read_ari_ili(paths.data)

ari_df["truth_date"] = pd.to_datetime(ari_df["truth_date"])
ili_df["truth_date"] = pd.to_datetime(ili_df["truth_date"])

ari_df = ari_df.sort_values(["location", "truth_date"]).reset_index(drop=True)
ili_df = ili_df.sort_values(["location", "truth_date"]).reset_index(drop=True)

gap_ari = gap_summary(ari_df, disease="ARI", calendar_mode="data_range")
gap_ili = gap_summary(ili_df, disease="ILI", calendar_mode="data_range")

ok_ari = gap_ari[
    (gap_ari["n_observed"] >= MIN_OBS) &
    (gap_ari["max_gap_length"] <= MAX_GAP)
]["location"].tolist()

ok_ili = gap_ili[
    (gap_ili["n_observed"] >= MIN_OBS) &
    (gap_ili["max_gap_length"] <= MAX_GAP)
]["location"].tolist()

ok_ari = sorted(ok_ari)
ok_ili = sorted(ok_ili)

print("ARI shape:", ari_df.shape)
print("ILI shape:", ili_df.shape)
print("ok_ari:", ok_ari)
print("ok_ili:", ok_ili)


# ============================================================
# 3. Calendar
# ============================================================

full_weeks = pd.date_range(
    start=pd.Timestamp("2014-01-05"),
    end=max(ari_df["truth_date"].max(), ili_df["truth_date"].max()),
    freq="W-SUN"
)

train_weeks = full_weeks[
    (full_weeks >= pd.Timestamp("2014-01-05")) &
    (full_weeks <= pd.Timestamp("2022-12-25"))
]

test_weeks = full_weeks[
    (full_weeks >= pd.Timestamp("2023-01-01")) &
    (full_weeks <= pd.Timestamp("2023-12-31"))
]

print("train weeks:", len(train_weeks), train_weeks.min(), "->", train_weeks.max())
print("test weeks :", len(test_weeks), test_weeks.min(), "->", test_weeks.max())


# ============================================================
# 4. Helpers
# ============================================================

def get_train_series(df_raw, location):
    """
    Imputed training series using the same project imputation function.
    """

    return impute_series_weekly(
        df_raw[df_raw["location"] == location][["truth_date", "value"]],
        calendar=train_weeks,
        trim_to_first_obs=True,
        max_gap_knn=8
    ).astype(float)


def get_raw_full_series(df_raw, location):
    """
    Raw full weekly series on full_weeks calendar.
    """

    return extract_raw_weekly_series(df_raw, location, full_weeks).astype(float)


def make_week_features(date):
    """
    Weekly seasonality features.
    Week 53, if present, is compressed to 52.
    """

    week = int(date.isocalendar().week)
    week = min(week, 52)

    return {
        "sin_week": np.sin(2 * np.pi * week / 52.0),
        "cos_week": np.cos(2 * np.pi * week / 52.0)
    }


def make_training_row(series, t, disease, location):
    """
    Build one supervised training row.

    t is the forecast origin index.
    Features use information available at t:
      - recent_0 = y_t
      - recent_1 = y_{t-1}
      - ...
      - seasonal_52 = y_{t-52}
      - ...
    Targets:
      - y_1 = y_{t+1}
      - ...
      - y_4 = y_{t+4}
    """

    row = {
        "disease": disease,
        "location": location
    }

    for lag in RECENT_LAGS:
        row[f"recent_{lag}"] = float(series.iloc[t - lag])

    for lag in SEASONAL_LAGS:
        row[f"seasonal_{lag}"] = float(series.iloc[t - lag])

    row.update(make_week_features(series.index[t]))

    for h in HORIZONS:
        row[f"y_{h}"] = float(series.iloc[t + h])

    return row


def build_pooled_global_training_set(ari_df, ili_df, ok_ari, ok_ili):
    """
    Build pooled training data over all valid diseases and countries.
    """

    rows = []

    disease_map = {
        "ARI": (ari_df, ok_ari),
        "ILI": (ili_df, ok_ili)
    }

    min_required_lag = max(SEASONAL_LAGS)

    for disease, (df_raw, locations) in disease_map.items():

        for loc in locations:

            series = get_train_series(df_raw, loc).dropna()

            # t must allow seasonal_55 and future y_{t+4}
            for t in range(min_required_lag, len(series) - MAX_H):

                try:
                    row = make_training_row(series, t, disease, loc)
                    rows.append(row)
                except Exception:
                    continue

    df_train = pd.DataFrame(rows).dropna().reset_index(drop=True)
    return df_train


def get_value_available_ffill(available_series, date):
    """
    Return value at a date using information available up to that date only.
    If value at date is missing, use previous available value.
    No future leakage.
    """

    if date not in available_series.index:
        return np.nan

    val = available_series.loc[date]

    if pd.notna(val):
        return float(val)

    hist = available_series.loc[:date].dropna()

    if len(hist) == 0:
        return np.nan

    return float(hist.iloc[-1])


def make_eval_row_from_available(available_series, origin, disease, location):
    """
    Build one evaluation feature row using only information available up to origin.
    Missing lags are filled using forward-fill from the past only.
    """

    row = {
        "disease": disease,
        "location": location
    }

    for lag in RECENT_LAGS:
        date = origin - pd.Timedelta(weeks=lag)
        row[f"recent_{lag}"] = get_value_available_ffill(available_series, date)

    for lag in SEASONAL_LAGS:
        date = origin - pd.Timedelta(weeks=lag)
        row[f"seasonal_{lag}"] = get_value_available_ffill(available_series, date)

    row.update(make_week_features(origin))

    return row


def build_mase_scales(df_raw, locations, disease_name):
    """
    MASE scale by disease-country from training only.
    """

    rows = []

    for loc in locations:

        train_series = get_train_series(df_raw, loc)

        rows.append({
            "disease": disease_name,
            "location": loc,
            "mase_scale": mase_scale_seasonal(train_series, m=M_SEASON)
        })

    return pd.DataFrame(rows)


def expected_points_from_truth(y_test, disease, location):
    """
    Independent audit of available test targets.
    Depends only on raw truth, not model predictions.
    """

    rows = []

    for h in HORIZONS:

        n_expected = 0

        for origin in test_weeks:

            target = origin + pd.Timedelta(weeks=h)

            if target not in y_test.index:
                continue

            if pd.notna(y_test.loc[target]):
                n_expected += 1

        rows.append({
            "disease": disease,
            "location": location,
            "h": int(h),
            "expected_actual_points": int(n_expected)
        })

    return pd.DataFrame(rows)


# ============================================================
# 5. Train global pooled RF
# ============================================================

pooled_train = build_pooled_global_training_set(
    ari_df=ari_df,
    ili_df=ili_df,
    ok_ari=ok_ari,
    ok_ili=ok_ili
)

print("pooled_train shape:", pooled_train.shape)
display(pooled_train.head())

pooled_train_enc = pd.get_dummies(
    pooled_train,
    columns=["disease", "location"],
    drop_first=False
)

target_cols = [f"y_{h}" for h in HORIZONS]
feature_cols = [c for c in pooled_train_enc.columns if c not in target_cols]

X_train = pooled_train_enc[feature_cols].copy()
y_train = pooled_train_enc[target_cols].copy()

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)

rf_model = MultiOutputRegressor(
    RandomForestRegressor(**RF_PARAMS)
)

rf_model.fit(X_train_scaled, y_train)

print("Training complete.")
print("n_features:", len(feature_cols))
print("features:", list(feature_cols))


# ============================================================
# 6. Correct rolling-origin evaluation
# ============================================================

def pooled_rf_rolling_eval(
    df_raw,
    locations,
    disease_name,
    model,
    scaler,
    feature_cols
):
    """
    Correct rolling-origin evaluation for global pooled RF.

    Important:
      - The model is fixed after training on 2014-2022.
      - At each origin in 2023, current observed truth is revealed if available.
      - Features use information available up to origin.
      - Missing recent values are filled using past-only forward fill.
      - Rows are stored for every target inside 2023.
      - Missing actuals are not filtered here.
    """

    all_rows = []
    all_expected = []

    for loc in locations:

        print(f"\nEvaluating {disease_name} {loc}")

        train_series = get_train_series(df_raw, loc)
        raw_full_series = get_raw_full_series(df_raw, loc)
        y_test = raw_full_series.reindex(test_weeks)

        expected_df = expected_points_from_truth(
            y_test=y_test,
            disease=disease_name,
            location=loc
        )
        all_expected.append(expected_df)

        available_series = pd.Series(index=full_weeks, dtype=float)
        available_series.loc[train_series.index] = train_series.values

        rows_loc = []

        for origin in test_weeks:

            # Reveal current week if observed
            if pd.notna(y_test.loc[origin]):
                available_series.loc[origin] = float(y_test.loc[origin])

            # Build feature row using only current/past information
            feat_row = make_eval_row_from_available(
                available_series=available_series,
                origin=origin,
                disease=disease_name,
                location=loc
            )

            X_row = pd.DataFrame([feat_row])

            # If even after past-only fill something is missing, prediction cannot be produced
            if X_row.drop(columns=["disease", "location"]).isna().any(axis=None):
                preds = np.repeat(np.nan, MAX_H)
            else:
                X_row = pd.get_dummies(
                    X_row,
                    columns=["disease", "location"],
                    drop_first=False
                )

                X_row = X_row.reindex(columns=feature_cols, fill_value=0)

                try:
                    X_scaled = scaler.transform(X_row)
                    preds = model.predict(X_scaled)[0]
                except Exception:
                    preds = np.repeat(np.nan, MAX_H)

            for h in HORIZONS:

                target = origin + pd.Timedelta(weeks=h)

                if target not in y_test.index:
                    continue

                y_true = y_test.loc[target]

                try:
                    y_pred = preds[h - 1]
                except Exception:
                    y_pred = np.nan

                rows_loc.append({
                    "origin": origin,
                    "target": target,
                    "disease": disease_name,
                    "location": loc,
                    "h": int(h),
                    "y": float(y_true) if pd.notna(y_true) else np.nan,
                    "pred": float(y_pred) if pd.notna(y_pred) else np.nan,
                    "model": MODEL_NAME
                })

        all_rows.append(pd.DataFrame(rows_loc))

    long_df = pd.concat(all_rows, ignore_index=True) if all_rows else pd.DataFrame()
    expected_df = pd.concat(all_expected, ignore_index=True) if all_expected else pd.DataFrame()

    return long_df, expected_df


pooled_res_ari, pooled_expected_ari = pooled_rf_rolling_eval(
    df_raw=ari_df,
    locations=ok_ari,
    disease_name="ARI",
    model=rf_model,
    scaler=scaler,
    feature_cols=feature_cols
)

pooled_res_ili, pooled_expected_ili = pooled_rf_rolling_eval(
    df_raw=ili_df,
    locations=ok_ili,
    disease_name="ILI",
    model=rf_model,
    scaler=scaler,
    feature_cols=feature_cols
)

pooled_long = pd.concat([pooled_res_ari, pooled_res_ili], ignore_index=True)
pooled_expected = pd.concat([pooled_expected_ari, pooled_expected_ili], ignore_index=True)

print("\npooled_res_ari:", pooled_res_ari.shape)
print("pooled_res_ili:", pooled_res_ili.shape)
print("pooled_long   :", pooled_long.shape)
print("pooled_expected:", pooled_expected.shape)


# ============================================================
# 7. Metrics and audit
# ============================================================

ari_scales = build_mase_scales(ari_df, ok_ari, "ARI")
ili_scales = build_mase_scales(ili_df, ok_ili, "ILI")
pooled_scales = pd.concat([ari_scales, ili_scales], ignore_index=True)

pooled_long = pooled_long.merge(
    pooled_scales,
    on=["disease", "location"],
    how="left"
)

summary_rows = []

for (disease, location, h, model_name), sub in pooled_long.groupby(["disease", "location", "h", "model"]):

    sub = sub.copy()
    sub = sub[sub["y"].notna()].copy()
    sub = sub[sub["pred"].notna()].copy()
    sub = sub[sub["mase_scale"].notna()].copy()

    if len(sub) == 0:
        continue

    m_mae = mae(sub["y"].values, sub["pred"].values)
    m_rmse = rmse(sub["y"].values, sub["pred"].values)
    scale = float(sub["mase_scale"].iloc[0])
    m_mase = mase_from_mae(m_mae, scale)

    summary_rows.append({
        "disease": disease,
        "location": location,
        "h": int(h),
        "model": model_name,
        "mae": float(m_mae),
        "rmse": float(m_rmse),
        "mase": float(m_mase),
        "n": int(len(sub))
    })

pooled_country_h = (
    pd.DataFrame(summary_rows)
    .sort_values(["disease", "location", "h"])
    .reset_index(drop=True)
)

scored_points = (
    pooled_country_h
    .groupby(["disease", "location", "h"], as_index=False)
    .agg(scored_points=("n", "sum"))
)

pooled_n_audit = pooled_expected.merge(
    scored_points,
    on=["disease", "location", "h"],
    how="left"
)

pooled_n_audit["scored_points"] = pooled_n_audit["scored_points"].fillna(0).astype(int)
pooled_n_audit["missing_due_to_model_or_pred_nan"] = (
    pooled_n_audit["expected_actual_points"] - pooled_n_audit["scored_points"]
)

print("\nN audit aggregated:")
display(
    pooled_n_audit
    .groupby(["disease", "h"], as_index=False)
    .agg(
        expected_actual_points=("expected_actual_points", "sum"),
        scored_points=("scored_points", "sum"),
        missing_due_to_model_or_pred_nan=("missing_due_to_model_or_pred_nan", "sum")
    )
)


# ============================================================
# 8. Summary tables
# ============================================================

def build_summary_tables(country_h_df, disease):

    sub = country_h_df[country_h_df["disease"] == disease].copy()

    country_summary = (
        sub.groupby(["location", "model"], as_index=False)
        .agg(
            mae=("mae", "mean"),
            rmse=("rmse", "mean"),
            mase=("mase", "mean"),
            n_points=("n", "sum")
        )
        .sort_values(["mase", "mae"])
        .reset_index(drop=True)
    )

    horizon_summary = (
        sub.groupby(["h", "model"], as_index=False)
        .agg(
            mae=("mae", "mean"),
            rmse=("rmse", "mean"),
            mase=("mase", "mean"),
            n_countries=("location", "nunique"),
            n_points=("n", "sum")
        )
        .sort_values(["h", "mase", "mae"])
        .reset_index(drop=True)
    )

    return country_summary, horizon_summary


pooled_ari_country, pooled_ari_horizon = build_summary_tables(pooled_country_h, "ARI")
pooled_ili_country, pooled_ili_horizon = build_summary_tables(pooled_country_h, "ILI")

print("\nARI pooled RF horizon summary")
display(pooled_ari_horizon)

print("\nILI pooled RF horizon summary")
display(pooled_ili_horizon)

print("\nARI pooled RF country summary")
display(pooled_ari_country)

print("\nILI pooled RF country summary")
display(pooled_ili_country)


# ============================================================
# 9. Save outputs
# ============================================================

pooled_train.to_csv(paths.results / "rf_global_all_infections_all_countries_training_rows.csv", index=False)

pooled_long.to_csv(paths.results / "rf_global_all_infections_all_countries_long.csv", index=False)
pooled_expected.to_csv(paths.results / "rf_global_all_infections_all_countries_expected_points.csv", index=False)
pooled_n_audit.to_csv(paths.results / "rf_global_all_infections_all_countries_n_audit.csv", index=False)

pooled_country_h.to_csv(paths.results / "rf_global_all_infections_all_countries_country_h.csv", index=False)

pooled_ari_country.to_csv(paths.results / "rf_global_all_infections_all_countries_ari_country.csv", index=False)
pooled_ari_horizon.to_csv(paths.results / "rf_global_all_infections_all_countries_ari_horizon.csv", index=False)

pooled_ili_country.to_csv(paths.results / "rf_global_all_infections_all_countries_ili_country.csv", index=False)
pooled_ili_horizon.to_csv(paths.results / "rf_global_all_infections_all_countries_ili_horizon.csv", index=False)

print("\nSaved pooled RF outputs to:", paths.results)

ARI shape: (6685, 4)
ILI shape: (10646, 4)
ok_ari: ['BE', 'BG', 'CZ', 'DE', 'EE', 'LT', 'RO', 'SI']
ok_ili: ['BE', 'CZ', 'EE', 'GR', 'IE', 'LT', 'NL', 'PL', 'RO', 'SI']
train weeks: 469 2014-01-05 00:00:00 -> 2022-12-25 00:00:00
test weeks : 53 2023-01-01 00:00:00 -> 2023-12-31 00:00:00
pooled_train shape: (6678, 16)


,disease,location,recent_0,recent_1,recent_2,recent_3,seasonal_52,seasonal_53,seasonal_54,seasonal_55,sin_week,cos_week,y_1,y_2,y_3,y_4
0,ARI,BE,1583.5,1555.1,1827.1,1875.5,1514.0,1441.3,1493.7,1407.5,-0.885456,0.464723,1617.4,1287.0,1320.1,1652.7
1,ARI,BE,1617.4,1583.5,1555.1,1827.1,1401.2,1514.0,1441.3,1493.7,-0.822984,0.568065,1287.0,1320.1,1652.7,1743.1
2,ARI,BE,1287.0,1617.4,1583.5,1555.1,1560.0,1401.2,1514.0,1441.3,-0.748511,0.663123,1320.1,1652.7,1743.1,1875.1
3,ARI,BE,1320.1,1287.0,1617.4,1583.5,1392.1,1560.0,1401.2,1514.0,-0.663123,0.748511,1652.7,1743.1,1875.1,1934.0
4,ARI,BE,1652.7,1320.1,1287.0,1617.4,1748.8,1392.1,1560.0,1401.2,-0.568065,0.822984,1743.1,1875.1,1934.0,1951.9


Training complete.
n_features: 24
features: ['recent_0', 'recent_1', 'recent_2', 'recent_3', 'seasonal_52', 'seasonal_53', 'seasonal_54', 'seasonal_55', 'sin_week', 'cos_week', 'disease_ARI', 'disease_ILI', 'location_BE', 'location_BG', 'location_CZ', 'location_DE', 'location_EE', 'location_GR', 'location_IE', 'location_LT', 'location_NL', 'location_PL', 'location_RO', 'location_SI']

Evaluating ARI BE

Evaluating ARI BG

Evaluating ARI CZ

Evaluating ARI DE

Evaluating ARI EE

Evaluating ARI LT

Evaluating ARI RO

Evaluating ARI SI

Evaluating ILI BE

Evaluating ILI CZ

Evaluating ILI EE

Evaluating ILI GR

Evaluating ILI IE

Evaluating ILI LT

Evaluating ILI NL

Evaluating ILI PL

Evaluating ILI RO

Evaluating ILI SI

pooled_res_ari: (1616, 8)
pooled_res_ili: (2020, 8)
pooled_long   : (3636, 8)
pooled_expected: (72, 4)

N audit aggregated:


,disease,h,expected_actual_points,scored_points,missing_due_to_model_or_pred_nan
0,ARI,1,409,409,0
1,ARI,2,401,401,0
2,ARI,3,393,393,0
3,ARI,4,385,385,0
4,ILI,1,513,513,0
5,ILI,2,503,503,0
6,ILI,3,493,493,0
7,ILI,4,483,483,0



ARI pooled RF horizon summary


...


ARI pooled RF country summary


,location,model,mae,rmse,mase,n_points
0,BE,rf_global_all_infections_all_countries,153.388289,216.834423,0.516802,202
1,DE,rf_global_all_infections_all_countries,182.134205,272.533481,0.569729,202
2,BG,rf_global_all_infections_all_countries,108.170489,180.686604,0.581425,198
3,SI,rf_global_all_infections_all_countries,232.226973,304.490088,0.605659,202
4,CZ,rf_global_all_infections_all_countries,132.683148,182.932939,0.694815,202
5,RO,rf_global_all_infections_all_countries,109.394240,142.758886,0.799326,186
6,EE,rf_global_all_infections_all_countries,67.138945,105.066346,0.993042,198
7,LT,rf_global_all_infections_all_countries,168.501587,241.090866,1.080169,198



ILI pooled RF country summary


,location,model,mae,rmse,mase,n_points
0,IE,rf_global_all_infections_all_countries,6.819924,13.078895,0.542368,198
1,CZ,rf_global_all_infections_all_countries,11.363253,23.460749,0.567753,202
2,NL,rf_global_all_infections_all_countries,10.170491,20.061713,0.580890,202
3,BE,rf_global_all_infections_all_countries,72.993336,114.105425,0.686767,202
4,PL,rf_global_all_infections_all_countries,74.870353,138.180794,0.850111,202
5,SI,rf_global_all_infections_all_countries,14.690670,33.180601,1.132864,202
6,LT,rf_global_all_infections_all_countries,21.064398,55.182148,1.168424,198
7,EE,rf_global_all_infections_all_countries,58.157749,84.494847,1.639933,198
8,GR,rf_global_all_infections_all_countries,263.438292,462.970562,1.867218,202
9,RO,rf_global_all_infections_all_countries,4.994299,12.286385,1.915524,186



Saved pooled RF outputs to: C:\Users\aolas\UNI PYTHON\TFG\results
